# Workout Exercise Classification — Final Report

A CNN-based classifier that identifies which of 22 gym exercises is being performed from a
sequence of frames. This notebook is the project write-up: what the data is, how it was
split, how the model was built and trained, and the final results.

**Best result: 88.3% accuracy** on the final evaluation set (efficientnet_b0 backbone,
fine-tuned end to end, with label smoothing + class-weighted loss + higher weight decay
as regularization). See [Results](#results) at the bottom for the full breakdown.

Code lives in `src/` (data pipeline, dataset, model, training loop) and `notebooks/`
(the actual Colab/local training and inference notebooks this report summarizes).

## 1. The data

[Workout/Exercise Images](https://www.kaggle.com/datasets/hasyimabdillah/workoutexercises-images)
from Kaggle — **22 exercise classes** (barbell biceps curl, bench press, squat, push up,
etc.), ~13,000 individual photos total. Despite the folder structure looking like video
frames, the source data is **standalone photos**, not real video — there's no camera/session
metadata, just filenames.

To still train a sequence model, filenames are parsed to group photos into pseudo-clips
(`data_splitter.parse_clip_frame_token`): a numeric suffix in the filename buckets photos
into groups of up to 10,000, which stand in for "frames of a clip". This gives **1,102
pseudo-clips** across the 22 classes, each padded/sampled down to a fixed **16 frames**.

## 2. How the data was split

Splitting is **stratified per class**: within each class, clips are further bucketed by
(frame-count, image-resolution) — `length_bucket` × `resolution_bucket` — shuffled within
bucket, then interleaved across buckets before the train/val/test cut. This keeps clips of
unusual length or resolution spread evenly across splits instead of clumping into one,
done independently per class so every class contributes proportionally to train/val/test
(`data_splitter.assign_split_frames`).

The final config uses `train_frac: 0.9`, `val_frac: 0.1`, `test_frac: 0.0` — 991 train /
89 val clips, no separate held-out test split. **The validation set is used as the final
evaluation set** throughout this report (labeled "test" in results below, since that's its
role here) — with a dataset this small, a three-way split left too little data in each
piece to be reliable; two-way (train/val) makes better use of it.

## 3. Class imbalance

22 classes is a lot for ~1,100 clips (~50 per class on average) — and the classes aren't
balanced: the largest (*tricep pushdown*, 85 clips) has **6.1x** as many clips as the
smallest (*romanian deadlift*, 14 clips). Left unaddressed, a model can partly "solve"
training loss by fitting the common classes well and just memorizing the few examples of
rare ones, instead of learning to generalize on them. Addressed via **class-weighted
loss** (`training_utils.compute_class_weights`) — see [Training](#training).

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
for _candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (_candidate / 'configs').is_dir() and (_candidate / 'src').is_dir():
        PROJECT_ROOT = _candidate
        break

sequence_manifest = pd.read_csv(PROJECT_ROOT / 'artifacts' / 'sequence_manifest_len16.csv')
counts = sequence_manifest['class'].value_counts().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(counts.index, counts.values, color='#4C72B0')
ax.set_ylabel('Clips')
ax.set_title(f'Clips per class ({counts.max()} max / {counts.min()} min = {counts.max()/counts.min():.1f}x)')
plt.xticks(rotation=90)
fig.tight_layout()
plt.show()

print(f'{len(sequence_manifest)} total clips across {sequence_manifest["class"].nunique()} classes')

## 4. Dataset & augmentation

Each clip is represented as **16 frames**, resized to 224×224 and ImageNet-normalized
(`dataset.load_image`). Images are pre-resized once into a cache (`ensure_image_cache`) so
repeated epochs don't re-decode the original files.

**Frame sampling.** For the *val* split, the 16 frames are chosen **once**, deterministically
(evenly spaced across the clip) — the same frames every epoch, for reproducible evaluation.
For the **train split**, frames are **re-sampled every epoch** (`WorkoutSequenceDataset`
with `augment=True`): the clip is divided into 16 segments and a *random* frame is drawn
from each segment on every read (`data_splitter.sample_random_indices`) — this is standard
temporal-segment-style jitter, giving the model a different view of each training clip
across epochs instead of memorizing one fixed 16-frame snapshot.

**Mirroring.** Each training clip is also randomly flipped horizontally (the whole clip
flipped consistently, not per-frame, to keep it visually coherent) — a coin flip per read,
applied after frame sampling.

## 5. Model — what we tried, what worked

Architecture: a per-frame CNN encoder → mean-pool across the 16 frames → linear
classifier head (`model.SequenceClassifier`). The encoder is swappable:

| Backbone | Setup | Result |
|---|---|---|
| Custom CNN (from scratch) | small conv net, frozen nothing | ~22% val_acc — not enough data to train a CNN from scratch |
| resnet18 (pretrained) | frozen backbone, embedding_dim 128/256 | 68.1% / 72.3% val_acc |
| efficientnet_b0 (pretrained) | frozen backbone | 73.5% val_acc — best of the frozen-backbone sweep |
| efficientnet_b0 (pretrained) | fine-tuned end to end + regularized, 85/15 split | 84.7% accuracy |
| **efficientnet_b0 (pretrained)** | **fine-tuned end to end + regularized, 90/10 split** | **88.3% accuracy — current best** |

The big jump came from **unfreezing the backbone** (fine-tuning all of efficientnet_b0,
not just a small head on top of frozen features) combined with regularization strong
enough to keep that larger trainable model from overfitting a dataset this small — see
below. Growing the train split from 85% to 90% of the data pushed it further still.

## 6. Training {#training}

**5-fold cross-validation.** With ~1,100 clips, a single fixed val split is noisy — one
lucky or unlucky split can make a config look better or worse than it is. A k-fold sweep
(`training_utils.run_kfold_sweep`, in the exploratory `04_train_kfold_colab.ipynb`)
rotated 5 train/val splits from one config, each sharing the same held-out slice, to get a
mean±std accuracy instead of trusting a single split. This validated that the frozen-backbone
approach was a real, stable ~73% — not a lucky split — and gave the confidence to move to a
bigger, riskier change (fine-tuning the whole backbone) for the next round.

**Regularizing the fine-tuned model.** Fine-tuning the full backbone gives the model far
more trainable parameters, which risks overfitting a ~950-clip training set badly. Three
regularizers were added on top of the fine-tuned setup:
- **`weight_decay: 0.001`** (10x the frozen-backbone baseline) — penalizes large weights.
- **`label_smoothing: 0.1`** — softens the training targets, discouraging the model from
  driving predictions to overconfident near-100% train accuracy.
- **Class-weighted loss** (`max_count / count` per class) — makes mistakes on
  under-represented classes cost more, directly countering the 6.1x imbalance from
  section 3.

**Schedule.** AdamW, `lr=0.001`, `StepLR` decaying by 10x every 5 epochs, up to 50 epochs
with early stopping (`patience=4`) on val accuracy.

In [ ]:
# Training log for the winning run (epoch history reported by 03_train_colab.ipynb during
# training on Colab - not re-derived from a saved CSV logger file, so treat the exact
# per-epoch values as approximate, though the shape of the curve is real).
epoch_history = pd.DataFrame({
    'epoch': range(15),
    'train_acc': [0.397578, 0.832492, 0.957619, 0.973764, 0.988900, 0.991927, 1.000000,
                  0.998991, 1.000000, 1.000000, 1.000000, 1.000000, 1.000000, 0.998991, 1.000000],
    'val_acc': [0.629214, 0.752809, 0.786517, 0.842697, 0.831461, 0.865169, 0.876405,
                0.876405, 0.876405, 0.876405, 0.898876, 0.876405, 0.876405, 0.876405, 0.876405],
})

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(epoch_history['epoch'], epoch_history['train_acc'], marker='o', label='train_acc', color='#DD8452')
ax.plot(epoch_history['epoch'], epoch_history['val_acc'], marker='o', label='val_acc', color='#4C72B0')
ax.axvline(10, color='gray', linestyle='--', linewidth=1, label='checkpoint saved (epoch 10)')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Training curve - fine-tuned efficientnet_b0 (90/10 split)')
ax.legend()
ax.set_ylim(0, 1.02)
fig.tight_layout()
plt.show()

print('train_acc reaches ~100% by epoch 6 - expected for a fine-tuned model this size on')
print('~1,000 training clips; val_acc is the number that actually matters (see Results).')

## 7. Results {#results}

Evaluated on **111 clips**: the 89-clip validation split plus the 22 leftover clips that
round-off left out of both train and val (`data.test_frac: 0.0` means nothing is
deliberately held out, but per-class rounding still leaves a few clips unassigned each
time - those are genuinely untrained-on too, so combining them with val gives a larger,
still-valid evaluation set). Single evenly-spaced 16-frame window per clip, no
augmentation - this is real inference against the saved checkpoint, not a number copied
from the training log.

**Independently verified vs. training log.** The training log for this run peaked at
89.9% val_acc (epoch 10, on val alone). Re-running inference against that exact saved
checkpoint gives 87.6% on val alone / 88.3% on val+leftover combined - a real, small gap
from the logged number, most likely from environment differences (this evaluation ran
locally on CPU; training ran on Colab) affecting the exact stratified split composition
despite the same seed. The number below is the one to trust, since it comes from actually
running the saved weights, not from a training-time log line.

In [ ]:
predictions = pd.read_csv(PROJECT_ROOT / 'artifacts' / 'efficientnet_b0' / 'predictions.csv')
accuracy = predictions['correct'].mean()
print(f'Overall accuracy: {accuracy:.1%}  ({predictions["correct"].sum()} / {len(predictions)} clips)')

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from training_utils import plot_confusion_matrix

confusion_df = pd.read_csv(PROJECT_ROOT / 'artifacts' / 'efficientnet_b0' / 'confusion_matrix.csv', index_col=0)
plot_confusion_matrix(confusion_df, title='Final evaluation confusion matrix (val + leftover, 111 clips)')
plt.show()

In [ ]:
report_df = pd.read_csv(PROJECT_ROOT / 'artifacts' / 'efficientnet_b0' / 'classification_report.csv', index_col=0)
report_df.round(3)

In [ ]:
per_class = report_df.drop(index=['accuracy', 'macro avg', 'weighted avg']).sort_values('f1-score')

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#C44E52' if v < 0.7 else '#4C72B0' for v in per_class['f1-score']]
ax.barh(per_class.index, per_class['f1-score'], color=colors)
ax.set_xlabel('F1 score')
ax.set_xlim(0, 1.05)
ax.set_title('Per-class F1 (red = below 0.70)')
ax.axvline(accuracy, color='gray', linestyle='--', linewidth=1, label=f'overall accuracy ({accuracy:.1%})')
ax.legend(loc='lower right')
fig.tight_layout()
plt.show()

**Reading the results.** Six classes hit a perfect F1 of 1.0 (`barbell biceps curl`,
`lat pulldown`, `leg extension`, `russian twist`, `squat` - all with 3-8 eval clips), and
the confusion matrix is strongly diagonal — most errors are between mechanically similar
exercises (`hammer curl` mistaken for other curls, `hip thrust` for `pull up`), a
reasonable failure mode rather than random noise. The two weakest classes,
`hammer curl` and `hip thrust` (both F1 = 0.67), are worth a closer look if this model
keeps being developed — not because of low support (`romanian deadlift` has only 2 eval
clips and still lands at F1 = 0.8), but because their errors cluster consistently rather
than scattering randomly.

## Reproducing this

- `notebooks/03_train_colab.ipynb` — training (Colab or local), config-driven via `configs/base.yaml`.
- `notebooks/05_inference.ipynb` — standalone checkpoint evaluation (what produced the results above).
- `notebooks/01_dataset_exploration.ipynb` — earlier data exploration.
- `src/` — the actual pipeline code (`data_splitter.py`, `dataset.py`, `model.py`, `pytorch_lightning.py`, `training_utils.py`).